# 01 - Data Inception

Author: Ahmed Shahzad

**Goal:** get to know both datasets, find problems in them, clean them, and save clean copies for the next notebooks.

| Dataset | Target (what we predict) | Rows | Task |
|---|---|---:|---|
| Maternal Health Risk (UCI) | `RiskLevel` (low / mid / high) | 1,014 | 3-class classification |
| HTRU2 pulsar candidates (UCI) | `is_pulsar` (0 / 1) | 17,898 | binary classification, imbalanced |

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

RAW = Path("data/raw")
OUT = Path("data/processed")
OUT.mkdir(parents=True, exist_ok=True)

RISK_ORDER = ["low risk", "mid risk", "high risk"]
PULSAR_COLS = [
    "IP_Mean", "IP_SD", "IP_Kurtosis", "IP_Skewness",   # integrated pulse profile
    "DM_Mean", "DM_SD", "DM_Kurtosis", "DM_Skewness",   # DM-SNR curve
]

# The maternal file starts with a byte-order mark, so we read it with utf-8-sig.
mat_raw = pd.read_csv(RAW / "Maternal_Health_Risk_Data_Set.csv", encoding="utf-8-sig")
# HTRU_2.csv has no header row: 8 feature columns, then the class label.
pul_raw = pd.read_csv(RAW / "HTRU_2.csv", header=None, names=PULSAR_COLS + ["is_pulsar"])

print("Maternal :", mat_raw.shape)
print("Pulsar   :", pul_raw.shape)

Maternal : (1014, 7)
Pulsar   : (17898, 9)


## 1. Maternal Health Risk - first look

In [2]:
display(mat_raw.head())
print()
mat_raw.info()

,Age,SystolicBP,DiastolicBP,BS,BodyTemp,HeartRate,RiskLevel
0,25,130,80,15.0,98.0,86,high risk
1,35,140,90,13.0,98.0,70,high risk
2,29,90,70,8.0,100.0,80,high risk
3,30,140,85,7.0,98.0,70,high risk
4,35,120,60,6.1,98.0,76,low risk



<class 'pandas.DataFrame'>
RangeIndex: 1014 entries, 0 to 1013
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Age          1014 non-null   int64  
 1   SystolicBP   1014 non-null   int64  
 2   DiastolicBP  1014 non-null   int64  
 3   BS           1014 non-null   float64
 4   BodyTemp     1014 non-null   float64
 5   HeartRate    1014 non-null   int64  
 6   RiskLevel    1014 non-null   str    
dtypes: float64(2), int64(4), str(1)
memory usage: 55.6 KB


**Column meaning (Maternal Health Risk)**

| Column | Meaning | Unit |
|---|---|---|
| `Age` | age of the woman | years |
| `SystolicBP` | upper blood pressure value | mmHg |
| `DiastolicBP` | lower blood pressure value | mmHg |
| `BS` | blood glucose level | mmol/L |
| `BodyTemp` | body temperature | degrees Fahrenheit |
| `HeartRate` | resting heart rate | beats per minute |
| `RiskLevel` | pregnancy risk level - **target** | low / mid / high |

In [3]:
print("Missing values per column:")
print(mat_raw.isna().sum().to_string())
print()
display(mat_raw.describe().T.round(2))
print()
print(mat_raw["RiskLevel"].value_counts().to_string())

Missing values per column:
Age            0
SystolicBP     0
DiastolicBP    0
BS             0
BodyTemp       0
HeartRate      0
RiskLevel      0



,count,mean,std,min,25%,50%,75%,max
Age,1014.0,29.87,13.47,10.0,19.0,26.0,39.0,70.0
SystolicBP,1014.0,113.20,18.40,70.0,100.0,120.0,120.0,160.0
DiastolicBP,1014.0,76.46,13.89,49.0,65.0,80.0,90.0,100.0
BS,1014.0,8.73,3.29,6.0,6.9,7.5,8.0,19.0
BodyTemp,1014.0,98.67,1.37,98.0,98.0,98.0,98.0,103.0
HeartRate,1014.0,74.30,8.09,7.0,70.0,76.0,80.0,90.0



RiskLevel
low risk     406
mid risk     336
high risk    272


### Problem 1: more than half of the rows are exact copies

The table above looks fine, but let's count duplicated rows.

In [4]:
n_dup = int(mat_raw.duplicated().sum())
print(f"Exact duplicate rows : {n_dup} of {len(mat_raw)} ({n_dup / len(mat_raw):.0%})")
print(f"Unique rows          : {len(mat_raw) - n_dup}")

Exact duplicate rows : 562 of 1014 (55%)
Unique rows          : 452


This matters a lot. If we split the data randomly into train and test sets, copies of the *same* row land on both sides. The model then "predicts" test rows it has already memorised, and the score looks much better than it really is. This is **data leakage**.

We remove all exact duplicates **before** any splitting.

### Problem 2: an impossible heart rate

`HeartRate` has a minimum of 7 beats per minute, which is not possible for a living person.

In [5]:
display(mat_raw[mat_raw["HeartRate"] < 40])

,Age,SystolicBP,DiastolicBP,BS,BodyTemp,HeartRate,RiskLevel
499,16,120,75,7.9,98.0,7,low risk
908,16,120,75,7.9,98.0,7,low risk


Both rows are the same record (a copy), and 7 is almost certainly a typing error (for example 70 or 77). We cannot know the true value, so we remove it instead of guessing.

### Problem 3 (kept): the same vitals with different labels

After removing exact copies, some rows still have *identical measurements* but a different `RiskLevel`.

In [6]:
uniq = mat_raw.drop_duplicates()
feat_cols = [c for c in uniq.columns if c != "RiskLevel"]
conflict = uniq[uniq.duplicated(subset=feat_cols, keep=False)]
print(f"Rows sharing identical vitals with another row : {len(conflict)}")
print(f"... of which carry conflicting risk labels      : {conflict.groupby(feat_cols)['RiskLevel'].nunique().gt(1).sum()} groups")
display(conflict.sort_values(feat_cols).head(8))

Rows sharing identical vitals with another row : 71
... of which carry conflicting risk labels      : 35 groups


,Age,SystolicBP,DiastolicBP,BS,BodyTemp,HeartRate,RiskLevel
543,12,90,60,7.5,102.0,66,low risk
588,12,90,60,7.5,102.0,66,mid risk
82,12,95,60,6.9,98.0,65,mid risk
251,12,95,60,6.9,98.0,65,low risk
561,13,90,65,7.5,101.0,80,low risk
610,13,90,65,7.5,101.0,80,high risk
531,15,120,80,7.5,98.0,70,mid risk
579,15,120,80,7.5,98.0,70,low risk


Six simple measurements cannot capture everything a doctor looks at, so two women with the same vitals can honestly get different labels. We **keep** these rows (they are real), but we should expect a ceiling on accuracy and say so in the final report.

## 2. Cleaning the maternal data

In [7]:
mat = mat_raw.drop_duplicates().reset_index(drop=True)
mat = mat[mat["HeartRate"] >= 40].reset_index(drop=True)
mat["RiskLevel"] = pd.Categorical(mat["RiskLevel"], categories=RISK_ORDER, ordered=True)

assert mat.duplicated().sum() == 0
assert mat.isna().sum().sum() == 0
assert (mat["SystolicBP"] > mat["DiastolicBP"]).all()

print(f"Rows: {len(mat_raw)} -> {len(mat)}")
print()
print(mat["RiskLevel"].value_counts().reindex(RISK_ORDER).to_string())

Rows: 1014 -> 451

RiskLevel
low risk     233
mid risk     106
high risk    112


## 3. HTRU2 pulsar candidates - first look

A **pulsar** is a rapidly spinning neutron star that sends out a beam of radio waves, like a lighthouse. Radio telescopes record thousands of "candidate" signals; nearly all of them are radio interference or noise, and only a few are real pulsars.

Each candidate is described by 8 numbers computed from two curves:

| Column | Meaning |
|---|---|
| `IP_Mean`, `IP_SD`, `IP_Kurtosis`, `IP_Skewness` | mean, standard deviation, excess kurtosis and skewness of the **integrated pulse profile** |
| `DM_Mean`, `DM_SD`, `DM_Kurtosis`, `DM_Skewness` | the same four statistics for the **DM-SNR curve** |
| `is_pulsar` | 1 = real pulsar, 0 = noise / interference - **target** |

In [8]:
display(pul_raw.head())
print()
print("Missing values:", int(pul_raw.isna().sum().sum()))
print("Duplicate rows:", int(pul_raw.duplicated().sum()))
print()
display(pul_raw.describe().T.round(2))

,IP_Mean,IP_SD,IP_Kurtosis,IP_Skewness,DM_Mean,DM_SD,DM_Kurtosis,DM_Skewness,is_pulsar
0,140.562500,55.683782,-0.234571,-0.699648,3.199833,19.110426,7.975532,74.242225,0
1,102.507812,58.882430,0.465318,-0.515088,1.677258,14.860146,10.576487,127.393580,0
2,103.015625,39.341649,0.323328,1.051164,3.121237,21.744669,7.735822,63.171909,0
3,136.750000,57.178449,-0.068415,-0.636238,3.642977,20.959280,6.896499,53.593661,0
4,88.726562,40.672225,0.600866,1.123492,1.178930,11.468720,14.269573,252.567306,0



Missing values: 0


Duplicate rows: 0



,count,mean,std,min,25%,50%,75%,max
IP_Mean,17898.0,111.08,25.65,5.81,100.93,115.08,127.09,192.62
IP_SD,17898.0,46.55,6.84,24.77,42.38,46.95,51.02,98.78
IP_Kurtosis,17898.0,0.48,1.06,-1.88,0.03,0.22,0.47,8.07
IP_Skewness,17898.0,1.77,6.17,-1.79,-0.19,0.20,0.93,68.10
DM_Mean,17898.0,12.61,29.47,0.21,1.92,2.80,5.46,223.39
DM_SD,17898.0,26.33,19.47,7.37,14.44,18.46,28.43,110.64
DM_Kurtosis,17898.0,8.30,4.51,-3.14,5.78,8.43,10.70,34.54
DM_Skewness,17898.0,104.86,106.51,-1.98,34.96,83.06,139.31,1191.00
is_pulsar,17898.0,0.09,0.29,0.00,0.00,0.00,0.00,1.00


In [9]:
counts = pul_raw["is_pulsar"].value_counts().sort_index()
print(counts.to_string())
print(f"\nPulsar share: {counts[1] / len(pul_raw):.2%}")

is_pulsar
0    16259
1     1639

Pulsar share: 9.16%


**What we learn**

- No missing values and no duplicate rows, so nothing needs cleaning.
- The classes are **very imbalanced**: only about 9% of candidates are real pulsars. A model that always says "not a pulsar" would be about 91% accurate and completely useless, so we will judge this model with precision, recall, F1 and PR-AUC, not accuracy.
- Several columns are heavily right-skewed (for example `DM_Mean` and `DM_Kurtosis`). These are real measurements of unusual signals, not typing errors, so we keep them. Tree-based models handle such columns without any special treatment.

In [10]:
pul = pul_raw.copy()
pul["is_pulsar"] = pul["is_pulsar"].astype(int)
assert pul.isna().sum().sum() == 0
print(pul.shape)

(17898, 9)


## 4. Save clean data

In [11]:
mat.to_csv(OUT / "maternal_clean.csv", index=False)
pul.to_csv(OUT / "pulsar_clean.csv", index=False)
print(sorted(p.name for p in OUT.iterdir()))

['maternal_clean.csv', 'pulsar_clean.csv']


## Summary

| | Maternal Health Risk | HTRU2 Pulsar |
|---|---|---|
| Raw rows | 1,014 | 17,898 |
| Missing values | none | none |
| Problems found | 562 exact duplicates (55%); 1 impossible heart rate (7 bpm); a few identical-vitals rows with different labels (kept) | none, but strongly imbalanced (about 9% pulsars) |
| Rows after cleaning | 451 | 17,898 |
| Saved to | `data/processed/maternal_clean.csv` | `data/processed/pulsar_clean.csv` |

Next: `02_eda.ipynb`.